In [1]:
pip install torch transformers huggingface_hub matplotlib seaborn numpy accelerate


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from huggingface_hub import login
import json


# Check activate for text generation tasks

In [4]:
import repetition_neurons
import utils
import torch
from huggingface_hub import login
import json
import os
global_seed = 42
repetition_neurons.seed_everything(global_seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = 'meta-llama/Llama-3.1-8B'

BASE_DIR = os.getcwd()
filename = f"repetition_neuron_data/{model_name[-10:]}.jsonl"
repetition_dataset = utils.load_data(filename)[:1000]
model, tokenizer = utils.load_model(model_name = model_name, seed=global_seed)
#sortedNeurons = repetition_neurons.find_sorted_neurons(model, tokenizer, base_data=repetition_dataset)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:820: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_attentions` is. When `return_dict_in_generate` is not `True`, `output_attentions` is ignored.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [17]:
ab_dataset = utils.load_data(filename)[1000:1100]

# Detect repetition as Hiroka's method

In [6]:

import re
import random
from transformers import GenerationConfig
from tqdm import tqdm
# Detect repetition if the input text includes the same n-gram k-times within r tokens at the same intervals.
def countOverlap(text, query):
    return len(re.findall(query, text, overlapped=True))

def getFirstAppearingIdx(text, query):
    return re.finditer(query, text).__next__().start(0)


def detectRepetition(line, n, r, k):
    # line should be list of idx
    SEP = ' '
    for i in range(0, len(line)-n+1):
        ngram = line[i:i+n]
        ngramStr = SEP.join(map(str, ngram))
        lineRange = line[max(0, i+n-r):i+n]
        lineRangeStr = SEP.join(map(str, lineRange))
        countRepInRange = countOverlap(lineRangeStr, ngramStr)

        if k <= countRepInRange:
            try:
                firstPosition = getFirstAppearingIdx(lineRangeStr, ngramStr)
                firstPosition = len(lineRangeStr[:firstPosition].split()) 
                firstPosition += max(0, i+n-r)

                lineRangeStr4second = SEP.join(map(str, line[firstPosition+1:i+n]))
                secondPosition = getFirstAppearingIdx(lineRangeStr4second, ngramStr)
                secondPosition = len(lineRangeStr4second[:secondPosition].split())
                secondPosition += firstPosition + 1

                lineRangeStr4third = SEP.join(map(str, line[secondPosition+1:i+n]))
                thirdPosition = getFirstAppearingIdx(lineRangeStr4third, ngramStr)
                thirdPosition = len(lineRangeStr4third[:thirdPosition].split())
                thirdPosition += secondPosition + 1

                if (secondPosition - firstPosition) == (thirdPosition - secondPosition):
                    return ngram, firstPosition, secondPosition, thirdPosition
            except:
                pass

    return [], -1, -1, -1

class Deactivator():
    def __init__(self, targetLayer, neuronIds, mode, lastN=0):
        self.neuronIds = neuronIds

        assert mode in ['last', 'all', 'lastN'], 'mode should be last or all'
        self.mode = mode
        self.lastN = lastN

        self.outputHandle = targetLayer.register_forward_hook(self.deactivate)

    def deactivate(self,model, input, output):
        if self.mode == 'last':
          output[0, -1, self.neuronIds] *= 0
        elif self.mode == 'all':
          output[0, :, self.neuronIds] *= 0
        elif self.mode == 'lastN':
          output[0, -self.lastN:, self.neuronIds] *= 0
        else:
          print(f'{self.mode=} cannot be recognized')
          pass
        return output

    def release(self):
        self.outputHandle.remove()

def convertNeuronsToDict(neurons):
    layer2neurons = {}
    for fn in neurons:
        i, j = fn
        if i not in layer2neurons:
            layer2neurons[i] = []
        layer2neurons[i].append(j)
    return layer2neurons

# Ablate repetition neurons


In [15]:
#import regex as re
from tqdm.notebook import tqdm  # or use tqdm.notebook if you're in a Jupyter environment

def run_segment_ablation_study_2phase(model, tokenizer, texts, sortedNeurons, 
                                      segment_ranges=None, neurons_to_ablate_list=None, verbose=True):
    """
    Run segment-wise ablation studies for both 'top' and 'random' modes.
    """
    if segment_ranges is None:
        segment_ranges = [(0, 0.2), (0.4, 0.6), (0.8, 1.0)]
    if neurons_to_ablate_list is None:
        neurons_to_ablate_list = [50, 100, 150, 200, 250] #[20, 50, 100, 180, 250] [50, 100, 250, 

    neurons_by_layer_position = {}
    total_layers = len(model.model.layers) 
    for neuron_info in sortedNeurons:
        layer_idx, neuron_idx = neuron_info['neuron']
        relative_position = layer_idx / total_layers  
        neurons_by_layer_position.setdefault(relative_position, []).append(neuron_info)

    all_results = {"top": {}, "random": {}}

    for mode in ["top", "random"]:
        if verbose:
            print(f"\nRunning experiments in '{mode}' mode...")
        for neurons_to_ablate in neurons_to_ablate_list:
            if verbose:
                print(f"\nProcessing deactivate with {neurons_to_ablate} neurons...")
            results = conduct_segment_ablation_2phase(
                model, tokenizer, texts, neurons_by_layer_position, segment_ranges, neurons_to_ablate,
                modeNr=mode, verbose=verbose)
            all_results[mode][neurons_to_ablate] = results

    return all_results


def conduct_segment_ablation_2phase(model, tokenizer, dataset, neurons_by_layer_position, 
                                    segment_ranges, neurons_to_ablate, modeNr="top", 
                                    N=50, verbose=True):
    assert modeNr in ['top', 'random'], 'selectMode should be top or random'
    if verbose:
        tqdm.write(f"Deactivate repetition neurons in each segment")
    
    texts = [{'ids': line['generatedIds']} for line in dataset]
    results = []


    for i, (start, end) in enumerate(tqdm(segment_ranges, desc="Processing segment ranges", unit="segment", position=0)):
        if verbose:
            tqdm.write(f"\nProcessing segment {i+1}: {start} -> {end}...")
        

        segment_neurons = []
        segment_layer_indices = set()  
        for layer_position, neurons in neurons_by_layer_position.items():
            if start <= layer_position < end:
                segment_neurons.extend(neurons)
                segment_layer_indices.add(int(layer_position * len(model.model.layers)))  # Map relative position to layer index

        if modeNr == "top":
            segment_neurons_sorted = sorted(segment_neurons, key=lambda x: x['diffs'], reverse=True)
            neurons_to_ablate_list = [neuron['neuron'] for neuron in segment_neurons_sorted[:neurons_to_ablate]]
        elif modeNr == "random":
            import random
            random.shuffle(segment_neurons)
            neurons_to_ablate_list = [neuron['neuron'] for neuron in segment_neurons[:neurons_to_ablate]]
        else:
            raise ValueError("Invalid mode. Choose 'top' or 'random'.")

        # Group neurons by their layer indices
        neurons_by_layer = {}
        for neuron in neurons_to_ablate_list:
            layer_idx, neuron_idx = neuron
            neurons_by_layer.setdefault(layer_idx, []).append(neuron_idx)

        # Deactivate neurons in the relevant layers
        hooks = []  
        for layer_idx in segment_layer_indices:
            if layer_idx in neurons_by_layer:
                hook = Deactivator(model.model.layers[layer_idx].mlp.act_fn, neurons_by_layer[layer_idx], 'all')
                hooks.append(hook)


        segment_results = []
        numRep = 0
        for text in tqdm(texts, desc=f"Segment {i+1}", unit="text", leave=False, position=1):
            ngram, firstPosition, secondPosition, thirdPosition = detectRepetition(text['ids'], n=10, r=100, k=3)
            initialInput = torch.LongTensor([text['ids'][:secondPosition]]).to(model.device)

            generationConfigGreedy = GenerationConfig(
                max_new_tokens=10 + 200 - N,
                do_sample=False,
                eos_token_id=model.config.eos_token_id,
                pad_token_id=model.config.eos_token_id
            )
            additionalOutputs = model.generate(initialInput, generation_config=generationConfigGreedy)
            

            ngram, firstPosition, secondPosition, thirdPosition = detectRepetition(additionalOutputs[0].tolist(), n=10, r=100, k=3)
            gens = additionalOutputs[0].tolist()
            if ngram:
                line = f'({i})REPL: ' + repr(tokenizer.decode(gens))
                segment_results.append(line)
                numRep += 1
            else:
                line = f'({i})NORE: ' + repr(tokenizer.decode(gens))
                segment_results.append(line)

        if verbose:
            tqdm.write(f'{segment_ranges[i]} : {numRep} repetition')
        results.append({
            "segment": f"{start}->{end}",
            "neurons_ablated": neurons_to_ablate_list,
            "results": segment_results,
            "numRep": numRep
        })
        for hook in hooks:
            hook.release()
    return results

In [18]:
import regex as re
sortedNeurons = repetition_neurons.find_sorted_neurons(model, tokenizer, base_data=repetition_dataset, seed=global_seed)
z = run_segment_ablation_study_2phase(model, tokenizer, ab_dataset, sortedNeurons)

Sorted Repetition Neurons


Compute activation: 100%|██████████| 25/25 [00:15<00:00,  1.64sequence/s]



Running experiments in 'top' mode...

Processing deactivate with 50 neurons...
Deactivate repetition neurons in each segment


Processing segment ranges:   0%|          | 0/3 [00:00<?, ?segment/s]


Processing segment 1: 0 -> 0.2...


...

(0, 0.2) : 2 repetition

Processing segment 2: 0.4 -> 0.6...


Segment 2:   0%|          | 0/2 [00:00<?, ?text/s]

(0.4, 0.6) : 2 repetition

Processing segment 3: 0.8 -> 1.0...


Segment 3:   0%|          | 0/2 [00:00<?, ?text/s]

(0.8, 1.0) : 2 repetition


In [19]:
segment_ranges = ['0->0.2', '0.4->0.6', '0.8->1.0']
for mode, mode_results in z.items():
    for nr_groups, segment_results in mode_results.items():
        #for segment
        for segment_result in segment_results:
            segment_range = segment_result["segment"]
            numRep = segment_result["numRep"]
            print(f"Deactivate  number of repnr: {nr_groups} ,segment: {segment_range}, numRep: {numRep}")

Deactivate  number of repnr: 50 ,segment: 0->0.2, numRep: 2
Deactivate  number of repnr: 50 ,segment: 0.4->0.6, numRep: 2
Deactivate  number of repnr: 50 ,segment: 0.8->1.0, numRep: 2
Deactivate  number of repnr: 100 ,segment: 0->0.2, numRep: 2
Deactivate  number of repnr: 100 ,segment: 0.4->0.6, numRep: 2
Deactivate  number of repnr: 100 ,segment: 0.8->1.0, numRep: 2
Deactivate  number of repnr: 150 ,segment: 0->0.2, numRep: 2
Deactivate  number of repnr: 150 ,segment: 0.4->0.6, numRep: 2
Deactivate  number of repnr: 150 ,segment: 0.8->1.0, numRep: 2
Deactivate  number of repnr: 200 ,segment: 0->0.2, numRep: 2
Deactivate  number of repnr: 200 ,segment: 0.4->0.6, numRep: 2
Deactivate  number of repnr: 200 ,segment: 0.8->1.0, numRep: 2
Deactivate  number of repnr: 250 ,segment: 0->0.2, numRep: 2
Deactivate  number of repnr: 250 ,segment: 0.4->0.6, numRep: 2
Deactivate  number of repnr: 250 ,segment: 0.8->1.0, numRep: 2
Deactivate  number of repnr: 50 ,segment: 0->0.2, numRep: 2
Deactiva

# Ablate induction heads

In [20]:
from utils import seed_everything
import induction_heads
def ab_hd_textgen(model, tokenizer, avg_scores, 
                  test_dataset, percent_list=None, 
                  seed=42,
                  N=50, verbose=True):
    print("Ablation induction heads ...")
    if verbose:
        tqdm.write(f"Deactivate heads")
    if percent_list is None:
        percent_list = [1, 3, 5, 7, 10]

    seed_everything(seed)
    texts = [{'ids': line['generatedIds']} for line in test_dataset]
    results = []
    num_layers = len(avg_scores)
    num_heads = avg_scores[0].shape[0]
    masks = induction_heads.build_head_masks(avg_scores, num_layers, num_heads, percent_list, random_seed=seed)

    results = {"induction": {}, "random": {}}
    for ab_type in ["induction", "random"]:
        #results[ab_type] = {}
        for i, p in enumerate(percent_list):
            # 1) block_config
            mask = masks[ab_type][p]  # Tensor[num_layers, num_heads]
            block_config = {
                layer: [h for h, v in enumerate(mask[layer].tolist()) if v == 0.0]
                for layer in range(num_layers)
            }
            #print(f"[DEBUG] {ab_type=} {p=} => ablate heads:", block_config)

            # 2) Disable Wo_h
            try:
                orig_blocks = induction_heads.disable_Wo_heads(model, block_config)
            except Exception as e:
                print(f"[ERROR] disable_Wo_heads failed at p={p}: {e}")
                orig_blocks = {}

            percent_results = []
            numRep = 0
            tmp_text = []
            try:
                for text in texts:
                    ngram, firstPosition, secondPosition, thirdPosition = detectRepetition(text['ids'], n=10, r=100, k=3)
                    initialInput = torch.LongTensor([text['ids'][:secondPosition]]).to(model.device)
                    generationConfigGreedy = GenerationConfig(
                        max_new_tokens=10 + 200 - N,
                        do_sample=False,
                        eos_token_id=model.config.eos_token_id,
                        pad_token_id=model.config.eos_token_id
                    )
                    additionalOutputs = model.generate(initialInput, generation_config=generationConfigGreedy)
                    

                    ngram, firstPosition, secondPosition, thirdPosition = detectRepetition(additionalOutputs[0].tolist(), n=10, r=100, k=3)
                    gens = additionalOutputs[0].tolist()
                    if ngram:
                        line = f'({i})REPL: ' + repr(tokenizer.decode(gens))
                        tmp_text.append(line)
                        numRep += 1
                    else:
                        line = f'({i})NORE: ' + repr(tokenizer.decode(gens))
                        tmp_text.append(line)
            except Exception as e:
                print(f"[ERROR] generation failed at p={p}: {e}")

            if verbose:
                tqdm.write(f'{ab_type}_{p} : {numRep} repetition')
            #print(results[ab_type])
            percent_results.append({
                "mode": ab_type,
                "percent": p,
                #"neurons_ablated": neurons_to_ablate_list,
                "results": tmp_text,
                "numRep": numRep
        })
            # 4) Restore Wo_h
            try:
                induction_heads.restore_Wo_heads(model, orig_blocks)
            except Exception as e:
                print(f"[ERROR] restore_Wo_heads failed at p={p}: {e}")
            results[ab_type][p] = percent_results
    return results

In [21]:
from tqdm.notebook import tqdm 
import regex as re
sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)
ab_head = ab_hd_textgen(model, tokenizer, sortedHeads, ab_dataset, seed=global_seed)

Generating sequences: 100%|██████████| 5/5 [00:02<00:00,  1.83it/s]


Ablation induction heads ...
Deactivate heads
induction_1 : 2 repetition
induction_3 : 2 repetition
induction_5 : 2 repetition
induction_7 : 2 repetition
induction_10 : 2 repetition
random_1 : 2 repetition
random_3 : 2 repetition
random_5 : 2 repetition
random_7 : 2 repetition
random_10 : 2 repetition
